# Human Activity Recognition Using Smartphones — Grupo 11
## Parte 1: descripción del problema y análisis exploratorio

**Integrantes:** Juan David Villota Córdoba, Sammir Alejandro Bolaños Luna  
**Curso:** Introducción al Aprendizaje Automático — UdeA, 2026-2

**Base de datos:** Human Activity Recognition Using Smartphones, UCI, https://doi.org/10.24432/C54S4K

Este notebook reproduce las tablas y la figura de la sección 2 del informe.

## 1. Descarga y carga de los datos

In [ ]:
import io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
with urllib.request.urlopen(URL) as r:
    z = zipfile.ZipFile(io.BytesIO(r.read()))

# El zip de UCI puede traer otro zip adentro ("UCI HAR Dataset.zip")
internos = [n for n in z.namelist() if n.endswith(".zip")]
if internos:
    z = zipfile.ZipFile(io.BytesIO(z.read(internos[0])))

def leer(sufijo, **kw):
    ruta = [n for n in z.namelist() if n.endswith(sufijo) and "Inertial" not in n][0]
    return pd.read_csv(z.open(ruta), sep=r"\s+", header=None, **kw)

features = leer("features.txt")[1].tolist()
etiquetas = leer("activity_labels.txt").set_index(0)[1].to_dict()
print("Número de características:", len(features))
print("Nombres de características repetidos:", len(features) - len(set(features)))
print("Actividades:", etiquetas)

In [ ]:
# Los nombres repetidos se vuelven únicos agregando un sufijo
vistos = {}
nombres = []
for f in features:
    vistos[f] = vistos.get(f, 0) + 1
    nombres.append(f if vistos[f] == 1 else f"{f}_{vistos[f]}")

partes = {}
for conj in ["train", "test"]:
    X = leer(f"X_{conj}.txt")
    X.columns = nombres
    X["actividad"] = leer(f"y_{conj}.txt")[0].map(etiquetas)
    X["sujeto"] = leer(f"subject_{conj}.txt")[0]
    X["conjunto"] = conj
    partes[conj] = X

df = pd.concat(partes.values(), ignore_index=True)
print("Dimensiones totales:", df.shape)
print("Entrenamiento:", partes["train"].shape[0], "| Prueba:", partes["test"].shape[0])
print("Sujetos en entrenamiento:", sorted(partes["train"]["sujeto"].unique().tolist()))
print("Sujetos en prueba:", sorted(partes["test"]["sujeto"].unique().tolist()))
print("¿Algún sujeto en ambos conjuntos?", bool(set(partes["train"]["sujeto"]) & set(partes["test"]["sujeto"])))

## 2. Datos faltantes, tipos y rango de valores

In [ ]:
X_all = df[nombres]
print("Valores faltantes (NaN):", int(X_all.isna().sum().sum()))
print("Tipos de datos:", X_all.dtypes.value_counts().to_dict())
print("Mínimo global:", X_all.min().min(), "| Máximo global:", X_all.max().max())
print("Filas duplicadas:", int(X_all.duplicated().sum()))

## 3. Distribución de clases

In [ ]:
orden = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]
tabla = pd.crosstab(df["actividad"], df["conjunto"]).reindex(orden)[["train", "test"]]
tabla["total"] = tabla.sum(axis=1)
tabla["%"] = (100 * tabla["total"] / tabla["total"].sum()).round(1)
tabla

In [ ]:
# Filas para pegar dentro de la Tabla III del informe (main.tex)
nombres_es = {"WALKING": "Caminar", "WALKING_UPSTAIRS": "Subir escaleras",
              "WALKING_DOWNSTAIRS": "Bajar escaleras", "SITTING": "Sentado",
              "STANDING": "De pie", "LAYING": "Acostado"}
for act, f in tabla.iterrows():
    print(f"{nombres_es[act]} & {int(f['train'])} & {int(f['test'])} & {int(f['total'])} ({f['%']:.1f}) \\\\")

## 4. Figura para el informe

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

tr = partes["train"]
Z = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(tr[nombres]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].barh([nombres_es[a] for a in orden], tabla["total"].values)
ax[0].set_xlabel("Número de ventanas")
ax[0].set_ylabel("Actividad")
ax[0].set_title("(a) Distribución de clases")
for act in orden:
    m = tr["actividad"].values == act
    ax[1].scatter(Z[m, 0], Z[m, 1], s=3, alpha=0.5, label=nombres_es[act])
ax[1].set_xlabel("Componente principal 1")
ax[1].set_ylabel("Componente principal 2")
ax[1].set_title("(b) Proyección PCA (entrenamiento)")
ax[1].legend(markerscale=4, fontsize=8)
fig.tight_layout()
fig.savefig("fig_eda.png", dpi=200)
plt.show()

**Nota:** la PCA de esta figura es solo exploratoria. La reducción de dimensión formal (sección 5 de la guía) se hará en una entrega posterior, ajustando la PCA solo con los datos de entrenamiento de cada partición.